# Quote flow experiment: vehicle pre-fill

The working notebook behind the decision memo. Each step queries the ClickHouse tables directly, so you can see the numbers come together.
All data is synthetic, generated by `experiment/generate.py`.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
from experiment.generate import generate, START, BUG_DAYS, COMMISSION_RATE
from experiment.warehouse import Warehouse
from experiment import stats

data = generate(7)
wh = Warehouse("../.warehouse").load(data)
users = wh.sql("user_metrics", commission_rate=COMMISSION_RATE, start=START.isoformat())
users.head()

,user_id,arm,day,device,user_type,state,prior_quotes_90d,started,viewed,bound,cancelled_30d,retained_policy,vehicle_mismatch,retained_commission
0,U070000000,treatment,19,android,returning,CA,4,0,0,0,0,0,0,0.0
1,U070000001,treatment,13,android,new,TX,0,0,0,0,0,0,0,0.0
2,U070000002,treatment,14,web,new,FL,1,0,0,0,0,0,0,0.0
3,U070000003,control,18,android,new,CA,0,0,0,0,0,0,0,0.0
4,U070000004,control,12,android,new,CA,0,1,1,0,0,0,0,0.0


## 1. Before any result: is the split healthy?

The test was planned 50/50. A sample ratio mismatch means some users are missing from one arm, and every result after that is suspect.

In [2]:
counts = users.arm.value_counts()
print(counts.to_dict(), "p =", stats.srm_test(counts["control"], counts["treatment"]))

{'control': 99734, 'treatment': 97416} p = 1.7840664672183148e-07


In [3]:
daily = wh.sql("srm_daily")
daily.pivot(index="day", columns="device", values="treatment_share").head(6)

device,android,ios,web
day,,,
2026-08-03,0.3325,0.5017,0.5241
2026-08-04,0.3428,0.4954,0.5382
2026-08-05,0.3419,0.5074,0.5007
2026-08-06,0.4886,0.5095,0.5124
2026-08-07,0.4945,0.4931,0.5112
2026-08-08,0.4891,0.5103,0.4740


Android sits around 33% treatment for the first three days, then returns to 50%. That lines up with the Android logging bug fixed in the
day-4 release. Dropping those days for both arms keeps the comparison fair.

In [4]:
clean = users[users.day >= BUG_DAYS]
c = clean.arm.value_counts()
print(c.to_dict(), "p =", round(stats.srm_test(c["control"], c["treatment"]), 3))

{'treatment': 85847, 'control': 85617} p = 0.579


## 2. The funnel, straight from ClickHouse

In [5]:
wh.sql("funnel", window_start="2026-08-06")

,arm,users,started,viewed,bound
0,control,85617,44010,35150,4299
1,treatment,85847,47859,38947,4712


## 3. Primary metric: bind rate per user

In [6]:
def by_arm(df, col):
    a, b = df[df.arm == "control"][col], df[df.arm == "treatment"][col]
    return stats.two_proportions(col, int(a.sum()), a.size, int(b.sum()), b.size)

primary = by_arm(clean, "bound")
t = (clean.arm == "treatment").to_numpy()
adjusted, reduction = stats.cuped("bound", clean.bound, clean.prior_quotes_90d, t)
pd.DataFrame([primary.to_dict(), adjusted.to_dict()])[["metric", "control", "treatment", "diff", "ci_low", "ci_high", "p_value"]]

,metric,control,treatment,diff,ci_low,ci_high,p_value
0,bound,0.050212,0.054888,0.004676,0.002564,0.006788,0.000014
1,bound,0.050212,0.054888,0.004667,0.002560,0.006774,0.000014


In [7]:
print(f"CUPED variance reduction: {reduction:.1%}")

CUPED variance reduction: 0.5%


CUPED barely helps here. Past quoting tells us little about who will buy, so there's little variance to remove. Both estimates agree.

## 4. Guardrail: do the new policies stick?

In [8]:
wh.sql("cancellations", window_start="2026-08-06")

,arm,vehicle_mismatch,policies,cancelled_30d,cancel_rate,avg_premium
0,control,0,4208,252,0.0599,1623.28
1,control,1,91,24,0.2637,1604.77
2,treatment,0,4139,233,0.0563,1618.66
3,treatment,1,573,133,0.2321,1574.61


In [9]:
by_arm(clean[clean.bound == 1], "cancelled_30d").to_dict()

{'metric': 'cancelled_30d',
 'control': 0.0642009769713887,
 'treatment': 0.07767402376910017,
 'diff': 0.013473046797711469,
 'ci_low': 0.0028857831976431027,
 'ci_high': 0.024060310397779835,
 'relative': 0.20985734849044058,
 'p_value': 0.013016056306593476,
 'n_control': 4299,
 'n_treatment': 4712,
 'significant': True}

Cancellations rise, and the rise sits almost entirely in policies where the plate lookup returned the wrong trim.
That's the finding that changes the decision from "ship" to "fix, then ship".

## 5. Segments, corrected for multiple comparisons

In [10]:
rows = []
for col, vals in [("user_type", ["new", "returning"]), ("device", ["ios", "android", "web"])]:
    for v in vals:
        rows.append({"segment": f"{col}={v}", **by_arm(clean[clean[col] == v], "bound").to_dict()})
seg = pd.DataFrame(rows)
seg["survives_fdr"] = stats.benjamini_hochberg(seg.p_value.tolist())
seg[["segment", "diff", "ci_low", "ci_high", "p_value", "survives_fdr"]]

,segment,diff,ci_low,ci_high,p_value,survives_fdr
0,user_type=new,0.005044,0.002699,0.007389,0.000025,True
1,user_type=returning,0.003786,-0.000634,0.008207,0.093213,False
2,device=ios,0.003897,0.000739,0.007054,0.015612,True
3,device=android,0.005159,0.001827,0.008491,0.002403,True
4,device=web,0.005714,0.000274,0.011155,0.039545,True


## 6. Check against the truth

Because the data is synthetic, the true effect is known. The interval should cover it.

In [11]:
truth = data.truth[data.truth.day >= BUG_DAYS]
true_effect = (truth.p_bind_treatment - truth.p_bind_control).mean()
print(f"planted {true_effect:+.4f}, measured {primary.diff:+.4f} [{primary.ci_low:+.4f}, {primary.ci_high:+.4f}]")
wh.close()

planted +0.0046, measured +0.0047 [+0.0026, +0.0068]
